# Loan Approval: Predicting Credit Risk

**BAIM 3220 · Introduction to Python**

Competition: https://www.kaggle.com/competitions/3220-loan-approval

Work through this notebook alongside **`Loan-README.md`**. The section numbers here match the guide.

**How to use this notebook**

- Run each cell from the top with **Shift + Enter**.
- Fill in every cell marked `# TODO`. Replace each `...` with your code.
- Answer each 🧠 question in the markdown cell right below it, in your own words.

**Name:** &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**Kaggle username:**

## Step 1: Explore

### 1a. Load the data

This cell finds the data whether you're on Kaggle or your laptop. If it raises `StopIteration`, the CSV files aren't where it's looking; see Section 3 of the guide.

In [ ]:
import pandas as pd
from pathlib import Path

candidates = [Path("/kaggle/input/competitions/3220-loan-approval"), Path("."), Path("..")]
DATA_DIR = next(p for p in candidates if (p / "train.csv").exists())
print("Reading data from:", DATA_DIR.resolve())

train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

print("train:", train.shape, "  test:", test.shape)
train.head()

### 1b. Get to know the columns

In [ ]:
# TODO: show the column types and non-missing counts for train
...

In [ ]:
# TODO: show summary statistics for the numeric columns of train
...

In [ ]:
# TODO: count the missing values in each column of train
...

### 1c. Ask questions with `groupby`

Start with the overall rate of 1s, then break it down by group.

In [ ]:
# TODO: what fraction of loans have loan_status == 1?
...

In [ ]:
# TODO: the rate of 1s for each loan_grade
...

In [ ]:
# TODO: the rate of 1s by cb_person_default_on_file
...

In [ ]:
# TODO: the rate of 1s by person_home_ownership
...

In [ ]:
# TODO: ask at least one more question of your own.
# Ideas: loan_intent, or a numeric column cut into bins with pd.cut / pd.qcut.
...

Optional: a picture is often clearer than a table.

In [ ]:
import matplotlib.pyplot as plt

rate = train.groupby("loan_grade")["loan_status"].mean()
rate.plot(kind="bar", title="Share of loan_status = 1 by loan grade", ylabel="rate of 1s", rot=0)
plt.show()

🧠 **Question 1.** What did you learn from each of your `groupby` questions? Name the group with the highest and lowest rate in each.

*Your answer:*

🧠 **Question 2.** Based on your exploration, what do you think `loan_status = 1` represents? Why might renters look riskier than homeowners? Does a high rate for a group prove that being in that group *causes* the outcome?

*Your answer:*

## Step 2: Your first prediction (no ML yet)

### 2a. The "always 0" baseline

Before predicting anything, measure how accurate a model that learns nothing would be.

In [ ]:
from sklearn.metrics import accuracy_score

# TODO: what fraction of train rows have loan_status == 0?
...

### 2b. A one-rule baseline

Predict 1 for grades D–G and 0 otherwise. First check it on the training data, then build a submission.

In [ ]:
# TODO: create a 0/1 Series that is 1 when train's loan_grade is D, E, F, or G
rule_train = ...

accuracy_score(train["loan_status"], rule_train)

In [ ]:
# TODO: apply the same rule to the TEST rows, then save the submission
predictions = ...

submission = pd.DataFrame({
    "id": test["id"],
    "loan_status": predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()

### 2c. Your turn: combine two rules

Predict 1 when the grade is D–G **or** `loan_percent_income > 0.3`. Use `|` for "or", and put each condition in parentheses.

In [ ]:
# TODO: build the two-rule prediction for train and check its accuracy
rule2_train = ...

accuracy_score(train["loan_status"], rule2_train)

🧠 **Question 3.** Predicting 0 for every loan is about 86% accurate. Why is that number not impressive for this data? What score does a model have to beat before it's actually useful?

*Your answer:*

**Baseline Kaggle score:**

## Step 3: Prepare the data for a model

### 3a. Check for missing values in both files

In [ ]:
# TODO: total number of missing values in train and in test
...

### 3b. Choose features

In [ ]:
features = [
    "person_age", "person_income", "person_emp_length",
    "loan_amnt", "loan_int_rate", "loan_percent_income",
    "cb_person_cred_hist_length",
    "loan_grade", "cb_person_default_on_file",
]

### 3c. Turn text into numbers

`loan_grade` is ordered, so map A–G to 1–7. Map `cb_person_default_on_file` from N/Y to 0/1. Apply the **same** dictionaries to both `train` and `test`.

In [ ]:
grade_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

for df in [train, test]:
    # TODO: map loan_grade with grade_map, and cb_person_default_on_file with {"N": 0, "Y": 1}
    df["loan_grade"] = ...
    df["cb_person_default_on_file"] = ...

train[["loan_grade", "cb_person_default_on_file"]].head()

### 3d. Separate features and target

In [ ]:
# TODO: build X, y, and X_test
X = ...
y = ...
X_test = ...

# These should print 0 and 0. If not, a .map() missed a category.
print(X.isna().sum().sum(), X_test.isna().sum().sum())
print(X.shape, X_test.shape)

## Step 4: Fit a model

Fit a decision tree with `max_depth=4` and `random_state=42` on all of `X` and `y`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# TODO: create and fit the model
model = ...
...

Peek inside: these are the questions your tree learned to ask. (Remember `loan_grade` 4 = D.)

In [ ]:
from sklearn.tree import export_text

print(export_text(model, feature_names=features, max_depth=2))

In [ ]:
# TODO: predict 0 or 1 for the first 5 rows of X
...

## Step 5: Check your model honestly

### 5a. See overfitting happen

Fit a tree with **no** `max_depth` and score it on the same data it learned from.

In [ ]:
# TODO: fit an unlimited-depth tree on X, y and print its accuracy on X
deep_model = ...
...

### 5b. Hold out a validation set

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TODO: fit a max_depth=4 tree on X_train, y_train, then print training and validation accuracy
model = ...
...

print("Training accuracy:  ", ...)
print("Validation accuracy:", ...)

### 5c. Find the best depth

In [ ]:
results = []
for depth in range(1, 16):
    # TODO: fit a tree with this depth on X_train, y_train and compute both accuracies
    m = ...
    ...
    tr = ...
    va = ...
    results.append({"depth": depth, "train_acc": tr, "val_acc": va})

results = pd.DataFrame(results).set_index("depth")
results.round(4)

In [ ]:
results.plot(marker="o", title="Decision tree accuracy by max_depth", ylabel="accuracy")
plt.axhline(1 - y.mean(), color="gray", linestyle="--", label="always predict 0")
plt.legend()
plt.show()

best_depth = results["val_acc"].idxmax()
print("Best depth:", best_depth, "  validation accuracy:", round(results.loc[best_depth, "val_acc"], 4))

🧠 **Question 4.** In your own words, what is **overfitting**? Describe what happened to training accuracy and validation accuracy as the depth grew, and how you used that to choose a depth.

*Your answer:*

## Step 6: Predict and submit

Refit with your best depth on **all** the training data, then predict 0 or 1 for each test row.

In [ ]:
# TODO: refit on X, y with your best depth, then predict X_test
model = ...
...
predictions = ...

submission = pd.DataFrame({
    "id": test["id"],
    "loan_status": predictions
})
submission.to_csv("submission.csv", index=False)

print(submission.shape)   # should be (39098, 2)
print(submission["loan_status"].value_counts())
submission.head()

Now submit `submission.csv` (see Section 10 of the guide) and record your score.

**Decision tree validation accuracy:**

**Decision tree Kaggle score:**

## Improvement: try at least one idea

Pick one or more ideas from Section 12 of the guide. Change **one thing at a time** and compare validation accuracy to your tree's. Some starting points are below. Delete what you don't use.

### Idea A: one-hot encode `person_home_ownership` and `loan_intent`

In [ ]:
# TODO (optional): one-hot encode the two remaining text columns, add the new
# columns to your features, and re-check validation accuracy with your best depth
train_oh = pd.get_dummies(train, columns=["person_home_ownership", "loan_intent"], dtype=int)
test_oh  = pd.get_dummies(test,  columns=["person_home_ownership", "loan_intent"], dtype=int)

features_oh = ...

### Idea B: a random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO (optional): fit a random forest on X_train, y_train and report validation accuracy
...

### Idea C: gradient boosting and new features

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# TODO (optional): try HistGradientBoostingClassifier, and/or create new features
...

### Submit your best model

In [ ]:
# TODO (optional): refit your best model on ALL training rows and overwrite submission.csv
...

🧠 **Question 5.** What did you try, what was the validation accuracy before and after, and did it help? Did your Kaggle score move the same way?

*Your answer:*

**Best Kaggle score:**

---
## Checklist before you turn this in

- [ ] Every `# TODO` cell runs without errors (**Kernel → Restart & Run All**)
- [ ] Questions 1–5 are answered in your own words
- [ ] Your baseline, decision tree, and best Kaggle scores are recorded above
- [ ] You uploaded a leaderboard screenshot to Canvas